# Sorting as a linear program

Author: [Martin Benning](m.benning@qmul.ac.uk)

Date: 25.01.2020

This is a little script to demonstrate how to sort an array $x$ (in ascending order) by finding the permutation matrix $P \in [0, 1]^{n \times n}$ via the linear program

$$ \min_{P \in \mathbb{R}^{n \times n}} - \langle w, Px \rangle \quad \text{subject to} \quad P_{ij} \in [0, 1] \, , \ \sum_{i = 1}^n P_{ij} = 1 \, , \ \sum_{j = 1}^n P_{ij} = 1 \quad \forall \, i, j \, , $$

for an array $w \in \mathbb{R}^n$ with monotonically increasing entries. Note that we can sort in descending order by choosing $w$ with entries that are monotonically decreasing. We begin by loading the NumPy library.

In [4]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

We set a seed for the random number generator to guarantee reproducibility.

In [5]:
np.random.seed(13)

We create an array *x* with *no_of_samples* real entries that are instances of a normal distributed random variable with mean zero and variance one.

In [6]:
no_of_samples = 6
x = np.random.randn(no_of_samples)
print(x)

[-0.71239066  0.75376638 -0.04450308  0.45181234  1.34510171  0.53233789]


We are going to sort the array *x* with linear programming, based on mirror descent augmented Lagrangian approach for which we require the [softmax function](https://en.wikipedia.org/wiki/Softmax_function). We define the function **softmax_function** that takes the one argument *argument* and an optional argument *axis*. The argument *argument* is a NumPy array and acts as the argument for the softmax operation, while the argument *axis* specifies the dimension across which we perform the summation.

In [7]:
def softmax_function(argument, axis=None):
    if axis == None:
        output = np.exp(argument - np.max(argument))
        output = output / np.sum(output)
    else:
        output = np.exp(argument - np.expand_dims(np.max(argument, axis), axis))
        output = output / np.expand_dims(np.sum(output, axis), axis)
    return output

We solve the linear program by iteratively approximating the saddle point of the augmented Lagrangian

$$ \mathcal{L}_{\delta}(P, Q; M) = - \frac12 \langle w, (P + Q)x \rangle + \chi_{S_1}(P) + \chi_{S_2}(P) + \langle M, P - Q \rangle + \frac{\delta}{2} \|P - Q \|_{F}^2 $$

with the following mirror-descent variant of the alternating direction method of multipliers:

\begin{align*}
P^{k + 1} &= \arg\min_{P \in [0, 1]^{n \times n}} \mathcal{L}_{\delta}(P, Q^k; M^k) + D_{f_1}(P, P^k) \, , \\
Q^{k + 1} &= \arg\min_{Q \in [0, 1]^{n \times n}} \mathcal{L}_{\delta}(P^{k + 1}, Q; M^k) + D_{f_2}(Q, Q^k) \, , \\
M^{k + 1} &= \arg\max_{M \in \mathbb{R}^{n \times n}} \mathcal{L}_{\delta}(P^{k + 1}, Q^{k + 1}; M) - \frac{1}{2\delta} \| M - M^k \|^2 \, .
\end{align*}
Here $\chi_{S_1}(P)$ and $\chi_{S_1}(P)$ are the the characteristic functions
\begin{align*}
\chi_{S_1}(P) = \begin{cases} 0 & P \in S_1 \\ \infty & P \not\in S_1 \end{cases} \qquad \text{and} \qquad \chi_{S_2}(Q) = \begin{cases} 0 & Q \in S_2 \\ \infty & Q \not\in S_2 \end{cases} \, ,
\end{align*}
for the sets 
\begin{align*}
S_1 = \left\{ P \in [0, 1]^{n \times n} \, \left| \, \sum_{i = 1}^n P_{ij} = 1 \, , \, \forall j \right. \right\} \quad \text{and} \quad S_2 = \left\{ Q \in [0, 1]^{n \times n} \, \left| \, \sum_{j = 1}^n Q_{ij} = 1 \, , \, \forall i \right. \right\} \, ,
\end{align*}
and $D_f$ is the Bregman distance with respect to a function $f$, i.e.
\begin{align*}
D_f(x, y) = f(x) - f(y) - \langle \nabla f(y), x - y \rangle \, .
\end{align*}
The functions $f_1$ and $f_2$ are defined as
\begin{align*}
f_1(P) &:= \frac{1}{\tau} \sum_{i = 1}^n \sum_{j = 1}^n \left[ P_{ij} \log(P_{ij}) - P_{ij} \right] - \frac{\delta}{2} \| P \|^2_F \, ,\\
f_2(Q) &:= \frac{1}{\tau} \sum_{i = 1}^n \sum_{j = 1}^n \left[ Q_{ij} \log(P_{ij}) - Q_{ij} \right] - \frac{\delta}{2} \| Q \|^2_F \, .\\
\end{align*}
We initialise $P^0$ and $Q^0$ so that they satisfy the constraints and compute *no_of_iterations* iterations of the method outlined above for hyperparameters $\delta = 1$, $\tau = 1$ and weights $w = \left(1, 2, \ldots, n \right)^\top$.

In [8]:
no_of_iterations = 500
P = 1/no_of_samples*np.ones((no_of_samples, no_of_samples))
Q = P
M = np.zeros((no_of_samples, no_of_samples))
w = (np.arange(no_of_samples) + 1)
data = w.reshape(-1, 1) @ x.reshape(1, -1)
for counter in range(no_of_iterations):
    P = softmax_function(np.log(P) + 1/2*data - (P - (Q - M)), axis=0)
    Q = softmax_function(np.log(Q) + 1/2*data - (Q - (P + M)), axis=1)
    M = M + P - Q
    if (counter + 1) % 100 == 0:
        print('Iteration no. {i}'.format(i=counter + 1))
Px = (P > 1/2).astype(int) @ x.reshape(-1, 1)
print('Output = {s} after {i} iterations.'.format(s=Px.reshape(-1), i=counter+1))

Iteration no. 100
Iteration no. 200
Iteration no. 300
Iteration no. 400
Iteration no. 500
Output = [-0.71239066 -0.04450308  0.45181234  0.53233789  0.75376638  1.34510171] after 500 iterations.


We see that the rounded version of $P$ applied to $x$ sorts the entries of $x$.

In [9]:
print('The original (unsorted) array is {a}, the sorted array reads {s}.'.format(a=x, \
        s=Px.reshape(-1)))

The original (unsorted) array is [-0.71239066  0.75376638 -0.04450308  0.45181234  1.34510171  0.53233789], the sorted array reads [-0.71239066 -0.04450308  0.45181234  0.53233789  0.75376638  1.34510171].


We conclude by printing the rounded permutation matrices $P$ and $Q$ and see that they are identical.

In [10]:
print("P = {p},\n\nQ = {q}.".format(p = np.round(P).astype(int), q = np.round(Q).astype(int)))

P = [[1 0 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 0 1]
 [0 1 0 0 0 0]
 [0 0 0 0 1 0]],

Q = [[1 0 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 0 1]
 [0 1 0 0 0 0]
 [0 0 0 0 1 0]].


This is the end of the notebook.